In [ ]:
import os
import shutil
import subprocess

uv = shutil.which("uv")
if uv is None:
    get_ipython().run_line_magic("pip", "install -q uv")
    uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("Could not find uv after installing it.")

repo_path = "/content/Fooocus"
if not os.path.isdir(os.path.join(repo_path, ".git")):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/Aditya-Y-Gitte/Fooocus.git",
            repo_path,
        ],
        check=True,
    )

os.chdir(repo_path)
subprocess.run([uv, "python", "install", "3.10"], check=True)
venv_path = os.path.join(repo_path, ".venv")
subprocess.run([uv, "venv", "--clear", "--python", "3.10", venv_path], check=True)

python = os.path.join(venv_path, "bin", "python")
subprocess.run(
    [
        uv,
        "pip",
        "install",
        "--python",
        python,
        "pip",
        "pygit2==1.15.1",
        "packaging==24.1",
        "fastapi==0.103.2",
        "starlette==0.27.0",
        "pydantic==1.10.15",
    ],
    check=True,
)
subprocess.run([python, "--version"], check=True)
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["LAUNCH_LIVE_OUTPUT"] = "1"
env["PYTHONUNBUFFERED"] = "1"
print(
    f"FOOOCUS_LAUNCH_CONFIG: python={python}; MPLBACKEND={env['MPLBACKEND']}; "
    f"LAUNCH_LIVE_OUTPUT={env['LAUNCH_LIVE_OUTPUT']}",
    flush=True,
)
print("Forwarding Fooocus output below:", flush=True)
process = subprocess.Popen(
    [python, "entry_with_update.py", "--share", "--always-high-vram"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env,
    text=True,
    bufsize=1,
)
if process.stdout is None:
    raise RuntimeError("Could not capture Fooocus output.")
for output_line in process.stdout:
    print(output_line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, process.args)